In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
DIFFUSION_CONFIG_PATH = "config/diffusion_full_celeba.yaml"   # relative to repo root
DIFFUSION_CKPT_PATH = None   # None → load best_ckpt.pt from experiment output dir
UNCONDITIONAL = False    # True → skip VAE, pass z=0
N_SAMPLES = 4
GUIDANCE_SCALE = 1.0    # > 1.0 requires a CFG-trained model (cfg_uncond_prob > 0)
SEED = 42

# DDPM-only options (ignored when noise_process is 'flow' or 'vpsde')
SAMPLER = "ddpm"   # 'ddpm' or 'ddim'
ETA = 0.0          # ddim only: 0=deterministic, 1≈ddpm stochasticity

# Number of steps for ddim / flow / vpsde (ddpm always uses all T steps)
NUM_INFERENCE_STEPS = 100

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)

## Imports

In [ ]:
import os
import sys
import types
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from models.beta_vae import BetaVAE
from models.unet import UNet
from models.noise_process import DDPM, RectifiedFlow, VPSDE
from data_utils import build_dataloaders
from utils import make_noise_schedule, diffusion_sample, flow_sample, vpsde_sample, make_run_tag

## Load models

In [ ]:
with open(os.path.join(REPO_ROOT, DIFFUSION_CONFIG_PATH)) as f:
    diff_cfg = types.SimpleNamespace(**yaml.safe_load(f))

diff_cfg.output_dir = os.path.join(REPO_ROOT, f"{diff_cfg.output_dir.lstrip('./')}_{make_run_tag(diff_cfg)}")
# betavae paths are absent in unconditional configs (keys commented out in yaml)
if hasattr(diff_cfg, 'betavae_config_path'):
    diff_cfg.betavae_config_path = os.path.join(REPO_ROOT, diff_cfg.betavae_config_path.lstrip("./"))
if hasattr(diff_cfg, 'betavae_ckpt_path'):
    diff_cfg.betavae_ckpt_path   = os.path.join(REPO_ROOT, diff_cfg.betavae_ckpt_path.lstrip("./"))
if not os.path.isabs(diff_cfg.data_dir):
    diff_cfg.data_dir = os.path.join(REPO_ROOT, diff_cfg.data_dir.lstrip("./"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- frozen β-VAE (skipped in unconditional mode) ---
if not UNCONDITIONAL:
    with open(diff_cfg.betavae_config_path) as f:
        vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))
    vae = BetaVAE(
        input_channels=vae_cfg.input_channels,
        input_height=vae_cfg.input_height,
        input_width=vae_cfg.input_width,
        latent_dim=vae_cfg.latent_dim,
        hidden_dim=vae_cfg.hidden_dim,
        beta=vae_cfg.beta,
        device=device,
    )
    vae_ckpt = torch.load(diff_cfg.betavae_ckpt_path, map_location=device)
    vae.load_state_dict(vae_ckpt["model_state_dict"])
    vae.eval()
    print(f"beta-VAE loaded (step {vae_ckpt['step']})")
else:
    print("Unconditional mode: skipping β-VAE")

# --- UNet ---
unet = UNet(
    in_channels=diff_cfg.in_channels,
    image_size=diff_cfg.image_size,
    model_channels=diff_cfg.model_channels,
    channel_mult=diff_cfg.channel_mult,
    num_res_blocks=diff_cfg.num_res_blocks,
    attention_resolutions=diff_cfg.attention_resolutions,
    dropout=0.0,
    latent_dim=diff_cfg.latent_dim,
)

if DIFFUSION_CKPT_PATH is None:
    ckpt_dir = os.path.join(diff_cfg.output_dir, "checkpoints")
    DIFFUSION_CKPT_PATH = os.path.join(ckpt_dir, "best_ckpt.pt")
    if not os.path.exists(DIFFUSION_CKPT_PATH):
        raise FileNotFoundError(f"best_ckpt.pt not found in {ckpt_dir}")

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet.load_state_dict(diff_ckpt["model_state_dict"])
unet = unet.to(device).eval()
print(f"UNet loaded: {DIFFUSION_CKPT_PATH}  (step {diff_ckpt['step']})")

## Load images

In [ ]:
import glob
from torchvision import transforms
from PIL import Image

exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
all_paths = sorted(
    p for p in glob.glob(os.path.join(diff_cfg.data_dir, "**", "*"), recursive=True)
    if os.path.splitext(p)[1].lower() in exts and "Zone.Identifier" not in p
)
sampled_paths = random.sample(all_paths, min(N_SAMPLES, len(all_paths)))

transform = transforms.Compose([
    transforms.Resize((diff_cfg.image_size, diff_cfg.image_size)),
    transforms.ToTensor(),
])
originals = torch.stack([transform(Image.open(p).convert("RGB")) for p in sampled_paths]).to(device)

### Noise schedule

In [ ]:
np_type = getattr(diff_cfg, 'noise_process', 'ddpm')
embed_scale = getattr(diff_cfg, 'num_timesteps', 1000)

if np_type == 'ddpm':
    noise_proc = DDPM(
        num_timesteps=diff_cfg.num_timesteps,
        beta_schedule=diff_cfg.beta_schedule,
        min_snr_gamma=getattr(diff_cfg, 'min_snr_gamma', 5.0),
        device=device,
    )
    schedule = make_noise_schedule(diff_cfg, device)
    print(f"DDPM  T={diff_cfg.num_timesteps}  beta_0={schedule['betas'][0]:.5f}  beta_T={schedule['betas'][-1]:.4f}")

elif np_type == 'flow':
    noise_proc = RectifiedFlow()
    schedule   = None
    print("RectifiedFlow — continuous ODE, t ∈ [0, 1]")

elif np_type == 'vpsde':
    noise_proc = VPSDE(
        beta_min=getattr(diff_cfg, 'vpsde_beta_min', 0.01),
        beta_max=getattr(diff_cfg, 'vpsde_beta_max', 5.0),
    )
    schedule   = None
    print(f"VPSDE  beta_min={noise_proc.beta_min}  beta_max={noise_proc.beta_max}")

else:
    raise ValueError(f"Unknown noise_process: {np_type!r}")

print(f"embed_scale={embed_scale}")

### Get beta-VAE latents

In [ ]:
with torch.no_grad():
    if UNCONDITIONAL:
        z = torch.zeros(originals.shape[0], diff_cfg.latent_dim, device=device)
        vae_recon = None
    else:
        z, _ = vae.encode(originals)
        vae_recon = torch.sigmoid(vae.decode(z)).cpu()

## One-step predictions

In [ ]:
n_vis = originals.shape[0]
t_vals = noise_proc.vis_t_vals()
n_rows = 1 + 2 * len(t_vals)

fig, axes = plt.subplots(n_rows, n_vis, figsize=(n_vis * 2, n_rows * 2), squeeze=False)

def _show(ax, img):
    img = img.cpu().permute(1, 2, 0).float().clamp(0, 1).numpy()
    ax.imshow(img.squeeze(-1) if img.shape[-1] == 1 else img,
              cmap='gray' if img.shape[-1] == 1 else None)
    ax.axis('off')

row_labels = ['x0']
for tv in t_vals:
    row_labels += [f'x_t  t={tv}', f'x_pred  t={tv}']

for col in range(n_vis):
    _show(axes[0, col], originals[col])

with torch.no_grad():
    for ti, t_val in enumerate(t_vals):
        t_tensor = torch.full((n_vis,), t_val, device=device, dtype=noise_proc.dtype)
        x_t, _ = noise_proc.corrupt(originals[:n_vis], t_tensor)
        pred = unet(x_t, noise_proc.embed_t(t_tensor, embed_scale), z[:n_vis])
        x0_pred = noise_proc.predict_x0(x_t, pred, t_tensor)
        row_xt = 1 + 2 * ti
        for col in range(n_vis):
            _show(axes[row_xt, col], x_t[col].clamp(0, 1))
            _show(axes[row_xt + 1, col], x0_pred[col].clamp(0, 1))

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=7, rotation=0, ha='right', va='center', labelpad=55)
fig.suptitle(f'One-step predictions ({np_type})', fontsize=9)
plt.tight_layout()
plt.show()

### Sample images and generate

In [ ]:
with torch.no_grad():
    if np_type == 'ddpm':
        generated = diffusion_sample(
            z, unet, schedule, diff_cfg, device,
            sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )
    elif np_type == 'flow':
        generated = flow_sample(
            z, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )
    elif np_type == 'vpsde':
        generated = vpsde_sample(
            z, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )

generated = generated.cpu()
originals_cpu = originals.cpu()

In [ ]:
n = originals.shape[0]

if UNCONDITIONAL:
    rows = [originals_cpu, generated]
    labels = ["original", "diffusion (unconditional)"]
    title = "original  |  diffusion generated (unconditional)"
else:
    rows = [originals_cpu, vae_recon, generated]
    labels = ["original", "β-VAE recon", "diffusion"]
    title = "original  |  β-VAE reconstruction  |  diffusion generated"

fig, axes = plt.subplots(len(rows), n, figsize=(n * 2.5, len(rows) * 2.5))
fig.suptitle(title, fontsize=11)

for row_idx, (imgs, label) in enumerate(zip(rows, labels)):
    axes[row_idx, 0].set_ylabel(label, fontsize=9)
    for col_idx in range(n):
        axes[row_idx, col_idx].imshow(imgs[col_idx].permute(1, 2, 0).clamp(0, 1))
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.show()

### Denoising progression (verbose)

In [ ]:
z_single = z[:1]

with torch.no_grad():
    if np_type == 'ddpm':
        _, snapshots = diffusion_sample(
            z_single, unet, schedule, diff_cfg, device,
            sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )
    elif np_type == 'flow':
        _, snapshots = flow_sample(
            z_single, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )
    elif np_type == 'vpsde':
        _, snapshots = vpsde_sample(
            z_single, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )

n_snaps = len(snapshots)
fig, axes = plt.subplots(1, n_snaps, figsize=(n_snaps * 2, 2.8))

proc_label = SAMPLER.upper() if np_type == 'ddpm' else np_type.upper()
steps_label = (f"[{diff_cfg.num_timesteps} steps]"
               if np_type == 'ddpm' and SAMPLER == 'ddpm'
               else f"[{NUM_INFERENCE_STEPS} steps]")
fig.suptitle(f"{proc_label} progression  (left: noise → right: final)  {steps_label}", fontsize=11)

for ax, (t_idx, img) in zip(axes, snapshots):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"t={t_idx:.3f}" if isinstance(t_idx, float) else f"t={t_idx}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()